# 03 — ARIMA, SARIMA, and LightGBM

Compare the same 90 series and 28-day windows; LightGBM pools all series, while ARIMA/SARIMA fit each series separately.

## Imports

In [1]:
from pathlib import Path
from types import SimpleNamespace
import os, json, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display


In [2]:
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'data/calendar.csv').exists())
DATA = ROOT / 'data'
OUT = DATA / 'processed'
RESULTS = ROOT / 'reports/results'
FIGURES = ROOT / 'reports/figures'
for folder in [OUT, RESULTS, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.venv/matplotlib'))


'/Users/moha/Machine-Learning-Project/Project 1/.venv/matplotlib'

In [3]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':100, 'axes.spines.top':False, 'axes.spines.right':False})
COLORS = ['#2563eb', '#059669', '#ea580c', '#7c3aed']


## Load prepared arrays

In [4]:
sales = np.load(OUT / 'sales.npy', mmap_mode='r')
prices = np.load(OUT / 'prices.npy', mmap_mode='r')
metadata = pd.read_csv(OUT / 'metadata.csv')
calendar = pd.read_csv(OUT / 'calendar.csv', parse_dates=['date'])
config = json.loads((ROOT / 'configs/comparison.json').read_text())
data = SimpleNamespace(root=ROOT)


## ARIMA, SARIMA, and LightGBM

In [5]:
import joblib
import lightgbm as lgb
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
sample=pd.read_csv(OUT / 'comparison_sample.csv')
sample_rows=sample.row.to_numpy()
assert len(sample)==90 and sample.series_id.tolist()==metadata.series_id.iloc[sample_rows].tolist()
display(pd.Series(config).to_frame('setting'))


,setting
horizon,28
development_cutoffs,"[1857, 1885]"
test_cutoff,1913
history_days,728
training_stride,13
seed,42
n_estimators,200
num_leaves,31
learning_rate,0.05
baseline_floor,0.05


## Calendar features

In [6]:
CATEGORICAL = ['item_id_code','store_id_code','dept_id_code','cat_id_code',
               'state_id_code','weekday','month','event_1','event_2']


In [7]:
def calendar_features(rows, days):
    cal = calendar.iloc[days-1]
    meta = metadata.iloc[rows]
    f = meta[CATEGORICAL[:5]].reset_index(drop=True).astype('int16')
    f['weekday'] = cal.wday.to_numpy(dtype='int8')
    f['month'] = cal.month.to_numpy(dtype='int8')
    f['event_1'] = cal.event_name_1_code.to_numpy(dtype='int16')
    f['event_2'] = cal.event_name_2_code.to_numpy(dtype='int16')
    f['trend'] = (days-1).astype('float32')
    angle = 2*np.pi*cal.date.dt.dayofyear.to_numpy()/365.25
    f['annual_sin'], f['annual_cos'] = np.sin(angle).astype('float32'), np.cos(angle).astype('float32')
    f['snap'] = np.select([meta.state_id.to_numpy()==s for s in ['CA','TX','WI']],
                          [cal['snap_'+s].to_numpy() for s in ['CA','TX','WI']]).astype('int8')
    return f


## Lag and rolling features

In [8]:
def demand_features(rows, days):
    t = days-1
    f = {f'lag_{lag}':sales[rows,t-lag].astype('float32') for lag in [28,35,42,56]}
    window = sales[rows[:,None],t[:,None]-28-np.arange(28)]
    f['mean_7'] = window[:,:7].mean(axis=1,dtype='float32')
    f['mean_28'] = window.mean(axis=1,dtype='float32')
    f['std_28'] = window.std(axis=1,dtype='float32')
    f['zero_rate_28'] = (window==0).mean(axis=1).astype('float32')
    return pd.DataFrame(f)


## Features available at the forecast origin

In [9]:
def origin_features(rows, days, origins):
    rows,days,origins = map(np.asarray,(rows,days,origins))
    assert ((days-origins>=1)&(days-origins<=28)).all() and origins.min()>=56
    f = pd.concat([calendar_features(rows,days),demand_features(rows,days)],axis=1)
    recent = sales[rows[:,None],origins[:,None]-1-np.arange(28)]
    last_weekday = origins-1-((origins-days)%7)
    weekly = sales[rows[:,None],last_weekday[:,None]-7*np.arange(8)]
    f['horizon'] = (days-origins).astype('int8')
    f['origin_mean_7'] = recent[:,:7].mean(axis=1,dtype='float32')
    f['origin_mean_28'] = recent.mean(axis=1,dtype='float32')
    f['origin_zero_rate_28'] = (recent==0).mean(axis=1).astype('float32')
    f['weekday_baseline'] = weekly.mean(axis=1,dtype='float32')
    return f


## Training dates and forecast indices

In [10]:
def training_indices(cutoff):
    rows,days = [],[]
    start = cutoff-config['history_days']+1
    for offset in range(config['training_stride']):
        group = np.arange(offset,len(metadata),config['training_stride'])
        targets = np.arange(start+offset,cutoff+1,config['training_stride'])
        rows.append(np.repeat(group,len(targets)))
        days.append(np.tile(targets,len(group)))
    return np.concatenate(rows),np.concatenate(days)


In [11]:
def forecast_indices(sample_rows, cutoff):
    rows = np.repeat(np.asarray(sample_rows),config['horizon'])
    days = np.tile(np.arange(cutoff+1,cutoff+config['horizon']+1),len(sample_rows))
    return rows,days,np.full(len(rows),cutoff)


## Comparable error metrics

In [12]:
def rmsse_scales(history):
    scales=[]
    for values in history:
        active=np.flatnonzero(values)
        differences=np.diff(values[active[0]:].astype('float64')) if len(active) else []
        scales.append(float(np.mean(np.square(differences))) if len(differences) else 0.)
    return np.asarray(scales)


In [13]:
def score_forecast(prediction, actual, scales):
    assert prediction.shape==actual.shape and np.isfinite(prediction).all()
    assert (prediction>=0).all() and (scales>0).all()
    error=prediction.astype('float64')-actual
    rmsse=np.sqrt(np.mean(error**2,axis=1)/scales)
    return dict(mae=float(np.abs(error).mean()), rmse=float(np.sqrt(np.mean(error**2))),
                mean_rmsse=float(rmsse.mean()))


## Check RMSSE against hand calculations

In [14]:
toy=np.array([[0,0,2,4,3],[1,2,3,4,5]],dtype='float32')
np.testing.assert_allclose(rmsse_scales(toy),[2.5,1.0])
actual=np.ones((2,28))
assert score_forecast(actual,actual,np.ones(2))['mean_rmsse']==0
assert score_forecast(actual+1,actual,np.ones(2))['mean_rmsse']==1


## LightGBM settings

In [15]:
model_parameters=dict(objective='poisson',random_state=config['seed'],verbosity=-1,
    n_estimators=config['n_estimators'],learning_rate=config['learning_rate'],
    num_leaves=config['num_leaves'],max_bin=63,min_child_samples=100,
    reg_lambda=1.0,colsample_bytree=0.9,n_jobs=1)
MODEL_DIR=ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True)


## Train the pooled LightGBM correction

In [16]:
def fit_lightgbm(cutoff):
    started=time.perf_counter()
    rows,days=training_indices(cutoff)
    assert days.max()<=cutoff and len(np.unique(rows))==len(metadata)
    horizons=(rows*17+days*13)%28+1
    X=origin_features(rows,days,days-horizons)
    offsets=np.maximum(X.weekday_baseline.to_numpy(),config['baseline_floor'])
    y=sales[rows,days-1].astype('float32')/offsets
    fitted=lgb.LGBMRegressor(**model_parameters)
    fitted.fit(X,y,categorical_feature=CATEGORICAL,sample_weight=offsets)
    joblib.dump(fitted,MODEL_DIR / f'comparison_lightgbm_d{cutoff}.joblib')
    return fitted,time.perf_counter()-started


## Predict the matched sample with LightGBM

In [17]:
def predict_lightgbm(fitted, cutoff):
    rows,days,origins=forecast_indices(sample_rows,cutoff)
    X=origin_features(rows,days,origins)
    offsets=np.maximum(X.weekday_baseline.to_numpy(),config['baseline_floor'])
    prediction=np.maximum(fitted.predict(X)*offsets,0)
    return prediction.reshape(len(sample),config['horizon']).astype('float32')


## Fit one ARIMA or weekly SARIMA series

In [18]:
def fit_statistical(history, kind):
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter('always')
        if kind=='ARIMA':
            model=ARIMA(history,order=tuple(config['arima_order']),trend='n')
            fitted=model.fit(method_kwargs={'maxiter':config['maxiter']})
        else:
            model=SARIMAX(history,order=tuple(config['arima_order']),
                           seasonal_order=tuple(config['seasonal_order']),trend='n')
            fitted=model.fit(maxiter=config['maxiter'],disp=False)
    raw=np.asarray(fitted.forecast(config['horizon']))
    assert np.isfinite(raw).all()
    diagnostics=dict(converged=bool(fitted.mle_retvals.get('converged',True)),
                     warnings=len(captured),negative_raw_forecasts=int((raw<0).sum()))
    return np.maximum(raw,0),diagnostics


## Run a statistical model across the sample

In [19]:
def predict_statistical(cutoff, kind):
    predictions=np.empty((len(sample),config['horizon']),dtype='float32')
    diagnostics=[]
    for position,row in enumerate(sample_rows):
        history=sales[row,cutoff-config['history_days']:cutoff].astype('float64')
        prediction,info=fit_statistical(history,kind)
        predictions[position]=prediction
        diagnostics.append(dict(model=kind,cutoff=cutoff,series_id=sample.series_id.iloc[position],**info))
        if (position+1)%30==0:
            print(f'{kind}, d_{cutoff}: {position+1}/{len(sample)} series',flush=True)
    return predictions,diagnostics


## Save comparable scores and predictions

In [20]:
records,fit_diagnostics=[],[]
cutoffs=[*config['development_cutoffs'],config['test_cutoff']]
PREDICTIONS=DATA / 'interim'
PREDICTIONS.mkdir(exist_ok=True)

def record_result(name, cutoff, prediction, seconds):
    actual=sales[sample_rows,cutoff:cutoff+config['horizon']]
    scales=rmsse_scales(sales[sample_rows,:cutoff])
    phase='local_test' if cutoff==config['test_cutoff'] else 'development'
    records.append(dict(model=name,cutoff=cutoff,phase=phase,seconds=seconds,
                        **score_forecast(prediction,actual,scales)))
    np.save(PREDICTIONS / f'three_model_{name}_d{cutoff}.npy',prediction)
    pd.DataFrame(records).to_csv(RESULTS / 'three_model_scores.csv',index=False)


## Run the two development windows and the previous local test

In [21]:
for cutoff in cutoffs:
    fitted,elapsed=fit_lightgbm(cutoff)
    record_result('LightGBM',cutoff,predict_lightgbm(fitted,cutoff),elapsed)
    for name in ['ARIMA','SARIMA']:
        started=time.perf_counter()
        prediction,diagnostics=predict_statistical(cutoff,name)
        fit_diagnostics.extend(diagnostics)
        record_result(name,cutoff,prediction,time.perf_counter()-started)
        pd.DataFrame(fit_diagnostics).to_csv(RESULTS / 'three_model_fit_diagnostics.csv',index=False)
    print(pd.DataFrame(records).tail(3).to_string(index=False),flush=True)


/var/folders/lq/jhztqld97t305sjbgrvcllph0000gp/T/ipykernel_11087/2701393547.py:10: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  angle = 2*np.pi*cal.date.dt.dayofyear.to_numpy()/365.25


/var/folders/lq/jhztqld97t305sjbgrvcllph0000gp/T/ipykernel_11087/2701393547.py:10: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  angle = 2*np.pi*cal.date.dt.dayofyear.to_numpy()/365.25


ARIMA, d_1857: 30/90 series


ARIMA, d_1857: 60/90 series


ARIMA, d_1857: 90/90 series


...

ARIMA, d_1913: 90/90 series


SARIMA, d_1913: 30/90 series


SARIMA, d_1913: 60/90 series


SARIMA, d_1913: 90/90 series


   model  cutoff      phase   seconds      mae     rmse  mean_rmsse
LightGBM    1913 local_test 36.098230 0.939023 1.712525    0.695901
   ARIMA    1913 local_test  7.081100 0.951145 1.701184    0.686761
  SARIMA    1913 local_test 17.742566 0.950416 1.702237    0.686533


## Check sample identity and prediction integrity

In [22]:
scores=pd.DataFrame(records)
assert len(scores)==9 and set(scores.model)=={'ARIMA','SARIMA','LightGBM'}
for cutoff in cutoffs:
    for name in ['ARIMA','SARIMA','LightGBM']:
        values=np.load(PREDICTIONS / f'three_model_{name}_d{cutoff}.npy')
        assert values.shape==(90,28) and np.isfinite(values).all() and (values>=0).all()
display(scores)


,model,cutoff,phase,seconds,mae,rmse,mean_rmsse
0,LightGBM,1857,development,39.982452,1.032225,2.005789,0.765371
1,ARIMA,1857,development,7.356883,1.039993,2.013114,0.765080
2,SARIMA,1857,development,17.576379,1.036046,2.001691,0.763582
3,LightGBM,1885,development,36.253785,0.969272,1.826340,0.689823
4,ARIMA,1885,development,7.297108,0.980947,1.838413,0.680007
5,SARIMA,1885,development,16.775677,0.977878,1.837326,0.679086
6,LightGBM,1913,local_test,36.098230,0.939023,1.712525,0.695901
7,ARIMA,1913,local_test,7.081100,0.951145,1.701184,0.686761
8,SARIMA,1913,local_test,17.742566,0.950416,1.702237,0.686533


## Development ranking

In [23]:
ranking=scores[scores.phase=='development'].groupby('model').agg(
    mean_rmsse=('mean_rmsse','mean'),mae=('mae','mean'),rmse=('rmse','mean'))
ranking=ranking.sort_values('mean_rmsse')
display(ranking)
print('Development ranking only; the previously inspected local test is a retrospective check.')


,mean_rmsse,mae,rmse
model,,,
SARIMA,0.721334,1.006962,1.919509
ARIMA,0.722543,1.010470,1.925764
LightGBM,0.727597,1.000749,1.916064


Development ranking only; the previously inspected local test is a retrospective check.


## Record settings and package versions

In [24]:
import importlib.metadata
import platform
manifest=dict(config=config,sample_ids=sample.series_id.tolist(),python=platform.python_version(),
    versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','lightgbm','statsmodels']})
manifest['inputs']={p.name:{'bytes':p.stat().st_size,'mtime_ns':p.stat().st_mtime_ns}
                    for p in sorted(DATA.glob('*.csv'))}
(RESULTS / 'three_model_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')


3481